# Gás do Povo — Download: Pagamentos (Portal da Transparência)

**Fonte:** Portal da Transparência do Governo Federal / CGU  
**Dataset:** Gás do Povo — pagamentos mensais por voucher  
**URL base:** `https://portaldatransparencia.gov.br/download-de-dados/gas-do-povo/{AAAAMM}`  
**Cobertura:** novembro/2025 em diante (disponível a partir de 27/05/2026)  
**Atualização:** mensal  
**Dicionário de dados:** https://portaldatransparencia.gov.br/dicionario-de-dados/gas-do-povo

Cada requisição redireciona para:  
`https://dadosabertos-download.cgu.gov.br/PortalDaTransparencia/saida/gas-do-povo/{AAAAMM}_GasdoPovo.zip`

O arquivo ZIP contém um CSV com **dados nominais individuais** de beneficiários: nome, CPF (parcial), município, data de uso do voucher, CNPJ e nome do revendedor e valor pago.

In [ ]:
!pip install --upgrade pandas --quiet

In [ ]:
import os
import requests
import zipfile
import pandas as pd

In [ ]:
import datetime

# Gera dinamicamente as competencias desde nov/2025 ate o mes corrente
# (evita lista fixa que fica desatualizada — B4)
def gerar_competencias(inicio: str = "202511") -> list:
    ano_i, mes_i = int(inicio[:4]), int(inicio[4:])
    hoje = datetime.date.today()
    comp = []
    ano, mes = ano_i, mes_i
    while (ano, mes) <= (hoje.year, hoje.month):
        comp.append(f"{ano}{mes:02d}")
        mes += 1
        if mes > 12:
            mes = 1
            ano += 1
    return comp

COMPETENCIAS = gerar_competencias("202511")

URL_BASE  = "https://portaldatransparencia.gov.br/download-de-dados/gas-do-povo/{comp}"
RAW_DIR   = "raw/01_portal_transparencia"
CLEAN_DIR = "clean/01_portal_transparencia"

os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(CLEAN_DIR, exist_ok=True)

print(f"Competencias a baixar ({len(COMPETENCIAS)}): {COMPETENCIAS}")

In [ ]:
def baixar_competencia(comp: str) -> str:
    """Baixa o ZIP de uma competência e retorna o caminho do CSV extraído."""
    url      = URL_BASE.format(comp=comp)
    zip_path = os.path.join(RAW_DIR, f"{comp}_GasdoPovo.zip")

    print(f"Baixando competência {comp}...")
    with requests.get(url, stream=True, timeout=120, allow_redirects=True) as r:
        r.raise_for_status()
        with open(zip_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)
    print(f"  ZIP salvo: {zip_path} ({os.path.getsize(zip_path):,} bytes)")

    with zipfile.ZipFile(zip_path, "r") as zf:
        csvs = [n for n in zf.namelist() if n.lower().endswith(".csv")]
        zf.extractall(RAW_DIR)
    print(f"  Extraídos: {csvs}")
    return os.path.join(RAW_DIR, csvs[0]) if csvs else ""


caminhos = []
for comp in COMPETENCIAS:
    try:
        caminho = baixar_competencia(comp)
        if caminho:
            caminhos.append(caminho)
    except Exception as e:
        print(f"  AVISO: competência {comp} não disponível — {e}")

print(f"\nTotal de arquivos baixados: {len(caminhos)}")

In [ ]:
import hashlib

# Carrega e consolida todas as competencias em um unico DataFrame
frames = []
for caminho in caminhos:
    try:
        df_tmp = pd.read_csv(caminho, sep=";", encoding="latin1", dtype=str)
        frames.append(df_tmp)
    except Exception as e:
        print(f"Erro ao ler {caminho}: {e}")

if frames:
    df = pd.concat(frames, ignore_index=True)

    # Anonimizacao LGPD (B8): descarta nome, pseudonimiza CPF por hash SHA-256
    # O raw/ fica no .gitignore; o clean/ nunca deve conter dados nominais.
    if "NOME BENEFICIARIO" in df.columns:
        df = df.drop(columns=["NOME BENEFICIARIO"])
    if "CPF BENEFICIARIO" in df.columns:
        df["CPF_HASH"] = df["CPF BENEFICIARIO"].apply(
            lambda x: hashlib.sha256(str(x).encode()).hexdigest()[:16] if pd.notna(x) else ""
        )
        df = df.drop(columns=["CPF BENEFICIARIO"])

    print("Dimensoes consolidadas (apos anonimizacao):", df.shape)
    print(df.head(3))
else:
    print("Nenhum arquivo carregado.")

In [ ]:
if frames:  # guard contra NameError se nenhum arquivo foi baixado (C3)
    df_colunas = pd.DataFrame({
        "coluna": df.columns,
        "dtype": df.dtypes.astype(str),
    })
    df_colunas.to_csv(f"{CLEAN_DIR}/gas_do_povo_colunas.csv", index=False, encoding="utf-8-sig")

    df.sample(min(3000, len(df)), random_state=42).to_csv(
        f"{CLEAN_DIR}/gas_do_povo_amostra.csv", index=False, encoding="utf-8-sig"
    )

    df.to_csv(f"{CLEAN_DIR}/gas_do_povo_completo.csv", index=False, encoding="utf-8-sig")

    print(f"Arquivos salvos em '{CLEAN_DIR}/' (sem nome/CPF original — LGPD)")
    print(f"Dimensoes finais: {df.shape}")
else:
    print("Nenhum dado para salvar.")

## Referências

| Item | Detalhe |
|------|---------|
| Programa | Gás do Povo (Lei nº 15.348/2026) |
| Gestão | Ministério do Desenvolvimento e Assistência Social (MDS) |
| Pagamento | Caixa Econômica Federal (via voucher) |
| Fonte dos dados | Portal da Transparência — CGU |
| URL de download | `https://portaldatransparencia.gov.br/download-de-dados/gas-do-povo/{AAAAMM}` |
| Repositório direto | `https://dadosabertos-download.cgu.gov.br/PortalDaTransparencia/saida/gas-do-povo/` |
| Dicionário | https://portaldatransparencia.gov.br/dicionario-de-dados/gas-do-povo |
| Acesso via portal | https://portaldatransparencia.gov.br/gas-do-povo |